In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cse250a.modules.config import config
from cse250a.modules.core import TIME_CARDS, Gas, Network, Sensor, Time, Weather
from cse250a.modules.data import Discretizer, blocked_split, load_air_quality
from cse250a.modules.eval import (
    bin_means,
    concentration_error,
    expected_concentration,
    resolution_floor,
)
from cse250a.modules.models import DiscreteBN, HourOnlyBaseline

In [ ]:
df = load_air_quality( config.paths.data_dir / "air_quality" / "AirQualityUCI.csv")
print(df.shape)
df.isna().mean().round(3).rename("fraction missing").to_frame().T

## 2. Choose the variables, and drop rows with missing inputs

These are this experiment's choices; they live here in the notebook, not in `src/`.

- **target:** CO
- **weather:** #TODO: ADD THE RANGE OF THE WEATHER VALUES.
- **sensors:** #TODO: LIST THE SENSOR s1. s2.....

We drop rows with missing H, W because they are inputs to the model, without them we #TODO: add rational

In [ ]:
target = Gas.CO
weather = (Weather.T, Weather.RH)
sensors = tuple(Sensor)

before = len(df)
df = df.dropna(subset=[Time.HOUR, *weather])
print(f"dropped {before - len(df)} rows with missing weather; {len(df)} left")

## 3. Split

`blocked_split` shuffles whole **weeks** into train / val / test, so neighbouring hours (which are nearly identical) don't end up on both sides of the split.

In [ ]:
train_raw, val_raw, test_raw = blocked_split(df, seed=config.seed)
print({name: len(d) for name, d in [("train", train_raw), ("val", val_raw), ("test", test_raw)]})

## 4. Discretize

Fit the bin edges on **train only**, then apply the same edges to every split. 3 quantile bins → states 0 / 1 / 2 = low / medium / high, roughly a third of training hours each. `disc.cards` records how many bins each column got.

In [ ]:
binned = [target, *sensors, *weather]
disc = Discretizer(n_bins=3).fit(train_raw, binned)
train, val, test = (disc.transform(d) for d in (train_raw, val_raw, test_raw))

cards = {**disc.cards, **TIME_CARDS}
print("cards:", {str(k): v for k, v in cards.items() if k in binned or k == Time.HOUR})
train[binned].head()

## 5. Build the network

```
   hour ──► CO ──► S1 … S5 ◄── T, RH
```

`structure` maps each node to its parents. Every key gets a CPT; hour, T and RH appear only as parents, so they're inputs. `validate(train.columns)` catches typos, cycles, missing cards and missing columns before anything is fitted.

In [ ]:
network = Network(
    structure={
        target: (Time.HOUR,),
        **{s: (target, *weather) for s in sensors},
    },
    cards=cards,
).validate(train.columns)

# print("inputs:", network.inputs)
# print("\n")
print(network)

## 6. Fit

EM over every training row. CO is missing on some rows, and EM uses those too, weighting each possible CO value by its posterior. `result` keeps the objective per iteration. It must never go down (`is_monotone`).

In [ ]:
bn = DiscreteBN(network).fit(train)
r = bn.result
print(f"{r.n_iter} iterations, converged: {r.converged}, monotone: {r.is_monotone()}")

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(r.history, marker="o")
ax.set_xlabel("EM iteration"); ax.set_ylabel("log-lik + log-prior")
plt.show()

## 7. Data Check

In [ ]:
# P(CO | hour), first hours
bn.cpts[target].to_frame().round(2).head(6) 

In [ ]:
# P(S1 | CO, T, RH)
bn.cpts[Sensor.S1].to_frame().round(2).head(9)

## 8. Query: P(CO | H, W, S1,...,S5)

A query is **(target, evidence)**, passed at call time. The evidence is what a deployed device could see, hour, weather and the sensors, and **never a reference reading**. The model refuses one, so it can't read the answer.

`predict_proba` returns one row per hour and one column per bin (0 = low, 1 = medium, 2 = high); each row sums to 1. `[2]` would select P(high).

In [ ]:
evidence = (Time.HOUR, *weather, *sensors)

proba = bn.predict_proba(val, target, evidence)
compare = proba.round(2).assign(predicted=proba.idxmax(axis=1), true=val[target])
compare.head(8)

## 9. Evaluate against the baseline

Every phase has to beat **hour-of-day alone**. Both models get the same evidence; the baseline just only reads the hour. Scores use only the hours where the reference was observed.

We score on **val**. Use it for every comparison and choice (bins, structure, α), and evaluate on **test** once, at the end, for the report. Choosing by test scores would make them optimistic.

- **accuracy:** with 3 quantile bins, always guessing one bin scores about 1/3
- **mean log-likelihood:** the best single number for a probabilistic model (higher, closer to 0, is better)

In [ ]:
hour = HourOnlyBaseline(cards=cards).fit(train)
metrics = ["n", "accuracy", "macro_f1", "ber", "mean_log_likelihood"]

results = pd.DataFrame(
    {m.name: {k: m.evaluate(val, target, evidence)[k] for k in metrics} for m in (bn, hour)}
).T
results.round(3)

## 10. From bins back to a concentration

The model predicts **bins**, but the goal is the **concentration** of CO in mg/m³. The posterior gives a probability for each bin, and each bin stands for a typical concentration, so the natural estimate is the **probability-weighted average** of those typical values:

$$\widehat{\text{CO}}_t \;=\; \sum_{k} P(\text{bin}_k \mid \text{evidence}_t)\;\cdot\;\bar{c}_k$$

where $\bar{c}_k$ is the mean raw concentration of the **training** hours that fell in bin $k$. For example, if an hour's posterior is (0.1, 0.3, 0.6) and the bin means are (0.8, 1.9, 3.7) mg/m³, the estimate is 0.1·0.8 + 0.3·1.9 + 0.6·3.7 = 2.87 mg/m³. A confident posterior lands near one bin's mean; an unsure one lands in between, which is closer on average when the model can't tell.

This turns the BN into a concentration estimator we can score in real units (MAE, RMSE), against the raw reference readings.

In [ ]:
c_bar = bin_means(train_raw[Gas.CO], train[Gas.CO])
proba = bn.predict_proba(val, Gas.CO, evidence)          # compute once, reuse
estimate = expected_concentration(proba, c_bar)

In [ ]:
concentration_error(estimate, val_raw[Gas.CO])

In [ ]:
resolution_floor(val_raw[Gas.CO], val[Gas.CO], c_bar)